In [1]:
import numpy as np
import math

L, d_k, d_v =4,8,8
q = np.random.randn(L, d_k)
k = np.random.randn(L, d_k)
v = np.random.randn(L, d_v)

In [2]:
print("Q\n", q)
print("K\n", k)
print("V\n", v)


Q
 [[ 0.33880875  0.05949402  1.05117531  0.73003433 -1.06954268  0.50103112
  -1.61222675  0.60867229]
 [-0.50426609 -0.61952926  0.05204404  0.67818517 -0.87014223 -0.75795844
   1.08876767  0.32052121]
 [ 2.35759901 -0.41285421  1.20918354 -1.23889988 -0.12674169  0.688189
  -1.27934071  0.13689992]
 [ 1.35290086  0.37013855  0.15111561  1.77130733 -0.54244939 -1.62274933
   0.60926837 -1.25497272]]
K
 [[ 5.40121581e-02  7.17425596e-01  2.80361079e-01 -4.70769888e-02
   1.92392525e+00 -2.25701204e-01 -6.10219812e-01 -6.20977168e-02]
 [ 2.22076461e+00  9.53637017e-01 -6.59273850e-01  2.11472001e-01
   1.29624990e+00 -8.60801107e-01 -1.18608063e+00 -1.18074364e-01]
 [ 3.31648587e-01  1.17574697e+00 -1.58347378e+00 -7.79312556e-01
  -6.57909689e-01 -5.54560920e-01 -1.22610631e+00 -2.30218900e-01]
 [ 4.77559354e-01 -1.36206849e+00 -2.07120950e+00 -8.83402134e-01
  -4.16112411e-02  1.15872865e-03  5.89432740e-01  1.18452476e+00]]
V
 [[ 0.12339072 -1.60892414  0.08491813 -1.18990401 -0.79


$$
\text{self attention} = softmax\left(\frac{Q \cdot K^T}{\sqrt{d_k}} + M\right)V
$$




In [ ]:
#nhân ma trận Q . KT
np.matmul(q, k.T)

array([[-0.9034648 ,  0.29319918,  0.21132562, -2.9255781 ],
       [-2.67634616, -3.40623833, -1.92250319,  0.95287003],
       [ 0.60148977,  4.52733778,  0.58609473, -0.30764913],
       [-0.67363321,  3.75166067,  0.06287402, -2.84256422]])

In [ ]:
#tại sao lại cần phải chia cho sqrt(d_k) khi tính attention score?
q.var(), k.var(),np.matmul(q, k.T).var()

(np.float64(0.9552340510032704),
 np.float64(0.9563926136751125),
 np.float64(4.713814454212921))

In [5]:
scaled = np.matmul(q, k.T) / math.sqrt(d_k)
q.var(), k.var(), scaled.var()

(np.float64(0.9552340510032704),
 np.float64(0.9563926136751125),
 np.float64(0.589226806776615))

In [6]:
scaled

array([[-0.31942304,  0.10366157,  0.07471489, -1.03434806],
       [-0.94623126, -1.20428711, -0.67970752,  0.33689043],
       [ 0.21265875,  1.60065562,  0.20721578, -0.10877039],
       [-0.23816531,  1.32641235,  0.02222932, -1.00499822]])

Masking

In [7]:
mask = np.tril(np.ones((L, L)))
mask

array([[1., 0., 0., 0.],
       [1., 1., 0., 0.],
       [1., 1., 1., 0.],
       [1., 1., 1., 1.]])

In [8]:
mask[mask == 0] = -np.inf
mask[mask == 1] = 0

In [10]:
mask

array([[  0., -inf, -inf, -inf],
       [  0.,   0., -inf, -inf],
       [  0.,   0.,   0., -inf],
       [  0.,   0.,   0.,   0.]])

In [11]:
scaled + mask

array([[-0.31942304,        -inf,        -inf,        -inf],
       [-0.94623126, -1.20428711,        -inf,        -inf],
       [ 0.21265875,  1.60065562,  0.20721578,        -inf],
       [-0.23816531,  1.32641235,  0.02222932, -1.00499822]])

$$
\text{softmax} = \frac{e^{x_i}}{\sum_j e^{x_j}}
$$

In [13]:
def softmax(x):
    return (np.exp(x).T / np.sum(np.exp(x), axis=1)).T

In [15]:
attention = softmax(scaled + mask)
attention

array([[1.        , 0.        , 0.        , 0.        ],
       [0.56415832, 0.43584168, 0.        , 0.        ],
       [0.16662813, 0.66764823, 0.16572364, 0.        ],
       [0.13258064, 0.63382237, 0.17201569, 0.0615813 ]])

In [ ]:
def softmax(x):
    return(np.exp(x).T/np.sum(np.exp(x), axis = 1)).T

def scaled_dot_Product_attention(q, k, v, mask = None):
    d_k = q.shape[-1]
    scaled = np.matmul(q, k.T) / math.sqrt(d_k)
    if mask is not None:
        scaled = scaled + mask
    attention = softmax(scaled)
    values = np.matmul(attention, v)
    return values, attention